# Continuous Mountain Car

In [1]:
from mountaincar_utils import test_car, env_mountaincar, display_frames_as_gif, ReplayMemory, goalAchieved
from IPython.display import HTML
import torch
torch.set_num_threads(1)

# # test the environment
# frames = test_car(env_mountaincar, 100)
# anim = display_frames_as_gif(frames)
# HTML(anim.to_jshtml())

## Network

Q-larning was intrinsically designed for discrete actions. The action is chosen based on the Q value of the current state, projected on the next state which yields the possible maximal Q value. So the network of Q-learning takes state as input and output the estimated Q value as output for each possible action.
For continuous actions, to adapt to this framework, we devide the continuous action into segments and each segment corresponds to an action. In the current example, we set 3 actions: a force to the left with a force step and one for the right and 0 for doing nothing.

So for the network below applied to the mountain car problem, the state has the dimenion 2: x position and the velocity of the car and the action has the dimension of 3: the force to the left, the force to the right and 0.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Qnet(nn.Module):
    """Network with inputs:
        - state: horizontal position and velocity of the car
       and output:
        - action: a float value that's added to the previous action.
    """
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, 64)
        self.fc2 = nn.Linear(64, 16)
        self.fc3 = nn.Linear(16, action_dim)

    def forward(self, state):
        out = torch.relu(self.fc1(state))
        out = torch.relu(self.fc2(out))
        return self.fc3(out)

## Agent



For training, the action:
 - use an exploration ratio to control how the agent acts during training. This can help skip the local minimum.
 - the network was used to estimate the force change to the previous action force. So the real force value should be computed based on the previous force.
 - at the end of the episode, the base action force value should be reset to 0. Otherwise, the agent will start from a wrong dynamic.
 - the force step should be between 0.2 and 0.4. A small step leads to training failure.

In [3]:
import numpy as np
import random

class DQLAgent:
    def __init__(self, env, batch_size=128, epsilon=1.0, epsilon_decay=1.0/3000, epsilon_min=0.1):
        self.env = env

        self.state_num = 2 # 2 state variables: position & velocity
        self.action_num = 3 # 3 actions: increase force by a step to left or right or 0
        self.model = Qnet(self.state_num, self.action_num)
        self.optimizer = torch.optim.SGD(self.model.parameters(), lr=0.001)

        self.force_step = 0.4 # change step to the previous force

        self.epsilon = epsilon #exploration rate
        self.epsilon_decay = epsilon_decay
        self.epsilon_final = epsilon_min

        self.batch_size = batch_size
        
        self.reset()


    def reset(self):
        """Reset the base action value after each episode"""
        self.action_value = 0.0
        self.action_id = 1.0


    def act(self, state, train=True):
        if np.random.random() < self.epsilon and train:
            self.action_id = random.randint(0, self.action_num-1)
        else:
            if isinstance(state, np.ndarray):
                state = torch.tensor(state, dtype=torch.float32)
            with torch.no_grad():
                self.action_id = torch.argmax(self.model(state)).detach().item()

        # Symmetrically map action index to a change in force
        mid = self.action_num // 2
        change = (self.action_id - mid) * self.force_step
        self.action_value += change

        # convert the scalar action to the shape expected by MountainCarContinuous
        env_input = np.clip(float(self.action_value), -1.0, 1.0)
        env_input = np.array([env_input], dtype=np.float32) # format to montain car input

        return env_input
    
    
    def update_epsilon(self):
        self.epsilon -= self.epsilon_decay
        if self.epsilon < self.epsilon_final:
            self.epsilon = self.epsilon_final

    
    def learn(self, plays, alpha=0.9, gamma=0.8):
        # sample the replay
        samples = plays.sample(self.batch_size)
        state, reward, action, nextState, done = samples

        # compute the q values
        q_values = self.model(state)

        # we copy the q values, then adapt them after
        q_targets = q_values.detach().clone() # [batch, action_dim]

        with torch.no_grad():
            qs = self.model(nextState).detach() # [batch, action_dim]
        q_nexts = torch.max(qs, dim=1).values.unsqueeze(1) # [batch, 1]

        # update the q values with the next state
        tmp = reward + gamma * q_nexts  * (1. - done)  # [batch, 1]
        
        q_targets = q_targets.scatter(1, action.to(torch.int32), tmp) # [batch, action_dim]
        
        # update network
        loss = F.mse_loss(q_values, q_targets)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return loss.detach().item()

## Training

In [9]:
# train loop
from collections import deque

epochs = 3000
memory = ReplayMemory(1000)
agent = DQLAgent(env_mountaincar, batch_size=128)

scores = []
losses = []
frames = []
recent_scores = deque(maxlen=100)

for e in range(epochs):
    # reset environment
    state, _ = env_mountaincar.reset() # reset environment and get initial state
    agent.reset() # reset agent's action value and action id

    currState = state.copy()
    done = False

    tot_loss = 0
    count = 0
    frame_num = 0
    tot_reward = 0

    # run an episode
    while not done:
        # choose action
        action = agent.act(state)
        
        # take action on env
        state, reward, terminated, truncated, info = env_mountaincar.step(action)
        done = terminated or truncated
        int_done = 1. if done else 0.
        frame_num += 1

        # Optional: for faster goal reaching
        # Add an aggressive kinetic/potential energy bonus
        # Reward the agent heavily for moving fast and climbing high
        energy_bonus = (state[1] ** 2) + (state[0] + 0.5)

        # If it reaches the goal flag (position >= 0.45), give a massive reward
        # reward += energy_bonus
        
        # add to replay memory
        memory.add([currState, reward, int(agent.action_id), state, int_done])

        if len(memory) >= agent.batch_size:
            # train the network
            loss = agent.learn(memory)
            tot_loss += loss
            count += 1

        currState = state.copy()
  
        tot_reward += reward

    tmp = tot_loss/count if count > 0 else 0
    losses = np.append(losses, tmp)
    frames.append(frame_num)
    recent_scores.append(tot_reward)

    # update epsilon
    agent.update_epsilon()

    # training progress message
    if (e+1)%100 == 0:
        print(f"epoche: {e+1}, reward: {sum(recent_scores)/100}, frames: {frame_num}, loss: {tot_loss/count:.6f}, epsilon: {agent.epsilon:.6f}")
    
    # early stopping if the goal is reached
    if goalAchieved(recent_scores, e):
        break

epoche: 100, reward: 3.1645999082569833, frames: 91, loss: 0.276636, epsilon: 0.966667
epoche: 200, reward: -10.94264008692216, frames: 163, loss: 0.184351, epsilon: 0.933333
epoche: 300, reward: 58.7533199086953, frames: 194, loss: 0.246748, epsilon: 0.900000
epoche: 400, reward: 66.72987990412238, frames: 189, loss: 0.265160, epsilon: 0.866667
epoche: 500, reward: 73.64459991207121, frames: 83, loss: 0.298908, epsilon: 0.833333
epoche: 600, reward: 68.81455992425924, frames: 94, loss: 0.336310, epsilon: 0.800000
epoche: 700, reward: 49.582079940815184, frames: 227, loss: 0.290833, epsilon: 0.766667
epoche: 800, reward: -36.79772003930011, frames: 999, loss: 0.250440, epsilon: 0.733333
epoche: 900, reward: -21.112160031713707, frames: 999, loss: 0.184677, epsilon: 0.700000
epoche: 1000, reward: -13.098640039476674, frames: 114, loss: 0.269017, epsilon: 0.666667
epoche: 1100, reward: 14.95787996217774, frames: 114, loss: 0.142306, epsilon: 0.633333
epoche: 1200, reward: 3.3371999727970

## Test

In [10]:
frames = test_car(env_mountaincar, 900, agent=agent)
anim = display_frames_as_gif(frames)
HTML(anim.to_jshtml())

125
